# Stage 4: Feature Engineering

Loads the cleaned dataset, engineers the composite `UNDERDIAGNOSIS_RISK`
label from three literature-grounded signals, defines the final feature
set, and creates the train/test split.

**Requires:** `processed/county_data_cleaned.csv`
**Produces:** `processed/df_labeled.csv`, `processed/X.csv`, `processed/y.csv`,
`processed/X_train.csv`, `processed/X_test.csv`, `processed/y_train.csv`,
`processed/y_test.csv`

In [1]:
# IMPORT LIBRARIES
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from google.colab import drive

In [2]:
# MOUNT DRIVE AND LOAD CLEANED DATASET

drive.mount("/content/drive")

FOLDER_NAME = "data"
DATASET_PATH = f"/content/drive/MyDrive/{FOLDER_NAME}"
PROCESSED_DIR = f"{DATASET_PATH}/processed"

df = pd.read_csv(f"{PROCESSED_DIR}/county_data_cleaned.csv", dtype={"FIPS": str})
print(f"✅ SUCCESS: loaded cleaned dataset ({df.shape[0]} rows, {df.shape[1]} cols)")

Mounted at /content/drive
✅ SUCCESS: loaded cleaned dataset (3144 rows, 31 cols)


In [3]:
# LABEL ENGINEERING THE TARGET VARIABLE
# No direct data on county-level PMOS underdiagnosis in any dataset
# Approach: Construct a composite binary label from three "signals" grounded in literature
# ICD rates excluded due to underestimation of true PMOS prevalence (Neven)

# Signal 1: High SVI (Silva)
# Signal 2: Low preventative healthcare engagement (Silva)
# Signal 3: High NCHS code (Ramphul)

# Social Vulnerablility + Access
# Composite Label: Signal 1 AND (Signal 2 OR 3)
# Two potential pathways to define underdiagnosis:
#   A: Vulnerable + disengaged from healthcare (Silva)
#   B: Vulnerable + geographically isolated (Ramphul)

# Signal 1 - in top-quartile social vulnerability
svi_high = df["RPL_THEMES"] >= df["RPL_THEMES"].quantile(0.75)

# Signal 2 - low preventative healthcare engagement
# Cervical screening (CERVICAL) unavailabile in PLACES 2025
# due to survey question change, replaced with mammography
checkup_col = "ANNUAL_CHECKUP"
mammography_col = "MAMMOGRAPHY"

preventative_low = (
    (df[checkup_col] <= df[checkup_col].quantile(0.25)) |
     (df[mammography_col] <= df[mammography_col].quantile(0.25))
)

# Signal 3: rural/periurban area (NCHS code >= 4)
rural_high = df["RURAL_CODE"] >= 4

# Composite binary label - returns 0 (low risk) or 1 (high risk)
df["UNDERDIAGNOSIS_RISK"] = (
    svi_high & (preventative_low | rural_high)
).astype(int)

# Check distribution, aim for 20-35% positive rate
counts = df["UNDERDIAGNOSIS_RISK"].value_counts()
positive_rate = counts[1] / len(df) * 100
print(f"High-risk counties (label = 1): {counts[1]}")
print(f"Low-risk counties (label = 0): {counts[0]}")
print(f"Positive rate: {positive_rate:.1f}%")

High-risk counties (label = 1): 671
Low-risk counties (label = 0): 2473
Positive rate: 21.3%


In [4]:
# DEFINE FEATURES(X) AND TARGET(Y)
# Exclude identifier columns, the target itself, and all RPL_THEMES
# composites (used to construct the label, so they would leak Signal 1)

leak_cols = ['UNDERDIAGNOSIS_RISK', 'FIPS', 'COUNTY', 'ST_ABBR',
             'RPL_THEMES', 'ANNUAL_CHECKUP', 'MAMMOGRAPHY', 'RURAL_CODE',
             'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4']

X = df.drop(columns=leak_cols)
y = df['UNDERDIAGNOSIS_RISK']


# Split the data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Features included ({X.shape[1]}): {X.columns.tolist()}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Features included (20): ['EP_NOVEH', 'EP_AFAM', 'EP_UNINSUR', 'EP_LIMENG', 'EP_POV150', 'EP_UNEMP', 'EP_NOHSDP', 'EP_HISP', 'EP_MUNIT', 'EP_HBURD', 'EP_DISABL', 'EP_MINRTY', 'DEPRESSION_PREV', 'DIABETES_PREV', 'FOOD_INSECURITY', 'HIGH_CHOL_PREV', 'OBESITY_PREV', 'REGION_Northeast', 'REGION_South', 'REGION_West']
X_train shape: (2515, 20)
X_test shape: (629, 20)
y_train shape: (2515,)
y_test shape: (629,)


In [5]:
# SAVE FEATURE-ENGINEERED ARTIFACTS FOR MODEL TRAINING

df.to_csv(f"{PROCESSED_DIR}/df_labeled.csv", index=False)
X.to_csv(f"{PROCESSED_DIR}/X.csv", index=False)
y.to_csv(f"{PROCESSED_DIR}/y.csv", index=False)
X_train.to_csv(f"{PROCESSED_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{PROCESSED_DIR}/X_test.csv", index=False)
y_train.to_csv(f"{PROCESSED_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{PROCESSED_DIR}/y_test.csv", index=False)

print(f"✅ SUCCESS: feature-engineered datasets saved to {PROCESSED_DIR}")

✅ SUCCESS: feature-engineered datasets saved to /content/drive/MyDrive/data/processed
